# AFC Claims Analysis

**Task 1** — Exploratory analysis and dashboard KPIs  
**Task 2** — Claim count and cost forecast for April–December 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'AFC Case Study Data Scientist_claims_data.csv'
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 20)

## 1. Data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['claim_date'])
df.info()

In [ ]:
df.describe()

In [ ]:
for col in ['vehicle_type', 'damage_category', 'fault_type', 'status']:
    print(df[col].value_counts().to_string())
    print()

In [ ]:
df.isna().sum().to_frame('missing').assign(
    pct=lambda x: (x['missing'] / len(df) * 100).round(1)
)

## 2. Preprocessing

In [ ]:
# estimated_cost_eur: impute by damage_category x vehicle_type group median
# cost varies significantly across both dimensions, so global imputation would be misleading
group_med_cost = df.groupby(['damage_category', 'vehicle_type'])['estimated_cost_eur'].transform('median')
cat_med_cost   = df.groupby('damage_category')['estimated_cost_eur'].transform('median')
df['estimated_cost_eur'] = df['estimated_cost_eur'].fillna(group_med_cost).fillna(cat_med_cost)

# repair_duration_days: wartend_auf_teile claims are structurally longer,
# so duration should be imputed within the same damage_category x status cell
group_med_dur = df.groupby(['damage_category', 'status'])['repair_duration_days'].transform('median')
cat_med_dur   = df.groupby('damage_category')['repair_duration_days'].transform('median')
df['repair_duration_days'] = df['repair_duration_days'].fillna(group_med_dur).fillna(cat_med_dur)

# description: not used in any model; leave empty rows as-is

# Derived columns
df['year']          = df['claim_date'].dt.year
df['month']         = df['claim_date'].dt.month
df['quarter']       = df['claim_date'].dt.quarter
df['cost_delta']    = df['actual_cost_eur'] - df['estimated_cost_eur']
df['cost_delta_pct']= df['cost_delta'] / df['estimated_cost_eur']

# storniert claims are real operational events but carry no realized cost.
# Kept in the main dataset; excluded from cost and duration analyses via explicit filter.
active = df[df['status'] != 'storniert'].copy()

## 3. Exploratory Analysis

### Is claim volume growing over time, and is there a seasonal pattern?

In [ ]:
monthly_all = df.set_index('claim_date').resample('MS')['claim_id'].count()
monthly_cancelled = (
    df[df['status'] == 'storniert']
    .set_index('claim_date')
    .resample('MS')['claim_id']
    .count()
    .reindex(monthly_all.index, fill_value=0)
)
monthly_active = monthly_all - monthly_cancelled

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(monthly_active.index, monthly_active, alpha=0.2, color='#2980b9')
ax.plot(monthly_active.index, monthly_active,
        color='#2980b9', linewidth=1.2, label='Active claims')
ax.plot(monthly_active.index, monthly_active.rolling(3, min_periods=1).mean(),
        color='#2980b9', linewidth=2.5, linestyle='--', label='3-month avg')
ax.plot(monthly_cancelled.index, monthly_cancelled,
        color='#c0392b', alpha=0.6, linewidth=1, label='Cancelled')
ax.set_ylabel('Claims')
ax.set_xlabel(None)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

### Which damage categories drive the most cost variability?

In [ ]:
cat_order = (
    active.groupby('damage_category')['actual_cost_eur']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=active, y='damage_category', x='actual_cost_eur',
            order=cat_order, ax=ax, linewidth=0.8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.set_xlabel('Actual cost (EUR)')
ax.set_ylabel(None)
plt.tight_layout()
plt.show()

### Which customers represent the highest combined frequency and cost risk?

In [ ]:
cust_summary = active.groupby('customer_id').agg(
    total_claims  =('claim_id', 'count'),
    avg_cost      =('actual_cost_eur', 'mean'),
    total_cost    =('actual_cost_eur', 'sum'),
).reset_index()

top5 = cust_summary.nlargest(5, 'total_cost')

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    cust_summary['total_claims'],
    cust_summary['avg_cost'],
    s=cust_summary['total_cost'] / 300,
    alpha=0.6, color='#2980b9', edgecolors='white', linewidths=0.5
)
for _, row in top5.iterrows():
    ax.annotate(
        row['customer_id'],
        (row['total_claims'], row['avg_cost']),
        xytext=(6, 4), textcoords='offset points', fontsize=8, color='#333'
    )
ax.set_xlabel('Total claims (2023–present)')
ax.set_ylabel('Avg actual cost per claim (EUR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
plt.tight_layout()
plt.show()

### How accurately do initial estimates track actual costs?

In [ ]:
delta_by_cat = (
    active.groupby('damage_category')['cost_delta_pct']
    .mean()
    .mul(100)
    .sort_values()
)
colors = ['#27ae60' if v <= 0 else '#e74c3c' for v in delta_by_cat]

fig, ax = plt.subplots(figsize=(8, 4))
delta_by_cat.plot(kind='barh', ax=ax, color=colors, width=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean cost vs estimate (%)')
ax.set_ylabel(None)
plt.tight_layout()
plt.show()

### Do high-volume customers have different fault-type profiles?

In [ ]:
top10_custs = active.groupby('customer_id')['claim_id'].count().nlargest(10).index

fault_mix = (
    active[active['customer_id'].isin(top10_custs)]
    .groupby(['customer_id', 'fault_type'])['claim_id']
    .count()
    .unstack(fill_value=0)
)
fault_mix_pct = fault_mix.div(fault_mix.sum(axis=1), axis=0)
if 'Eigenverschulden' in fault_mix_pct.columns:
    fault_mix_pct = fault_mix_pct.sort_values('Eigenverschulden', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
fault_mix_pct.plot(kind='barh', stacked=True, ax=ax, width=0.65)
ax.set_xlabel('Claim share')
ax.set_ylabel(None)
ax.set_xlim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 4. Dashboard KPIs

In [ ]:
# Claim frequency: annualised over each customer's observed window
cust_span = df.groupby('customer_id')['claim_date'].agg(['min', 'max'])
cust_span['months_active'] = ((cust_span['max'] - cust_span['min']).dt.days / 30.44).clip(lower=1)

cust_vol = df.groupby('customer_id').agg(
    total_claims=('claim_id', 'count'),
    cancelled   =('status', lambda x: (x == 'storniert').sum()),
)
cust_vol['cancellation_rate']    = cust_vol['cancelled'] / cust_vol['total_claims']
cust_vol['claim_frequency_rate'] = cust_vol['total_claims'] / cust_span['months_active'] * 12

resolved = active[active['status'] == 'abgeschlossen']
cust_resolved = resolved.groupby('customer_id').agg(
    avg_cost_per_claim  =('actual_cost_eur', 'mean'),
    avg_repair_duration =('repair_duration_days', 'mean'),
    estimation_accuracy =('cost_delta_pct', 'mean'),
)

kpis = cust_vol[['claim_frequency_rate', 'cancellation_rate']].join(cust_resolved)

In [ ]:
kpis.sort_values('avg_cost_per_claim', ascending=False).style.format({
    'claim_frequency_rate'  : '{:.1f}',
    'cancellation_rate'     : '{:.1%}',
    'avg_cost_per_claim'    : '€{:,.0f}',
    'avg_repair_duration'   : '{:.1f}',
    'estimation_accuracy'   : '{:+.1%}',
}).background_gradient(subset=['avg_cost_per_claim'], cmap='YlOrRd')

**KPI definitions**

| KPI | Formula | Unit |
|---|---|---|
| Claim Frequency Rate | claims ÷ months active × 12 | claims / year |
| Avg Cost per Resolved Claim | mean(actual_cost_eur), resolved only | EUR |
| Estimation Accuracy | mean((actual − estimated) / estimated) | % delta |
| Cancellation Rate | cancelled ÷ all claims | % |
| Avg Repair Duration | mean(repair_duration_days), resolved only | days |

For the sales team, **Avg Repair Duration** and **Avg Cost per Resolved Claim** land most directly — fleet operators track vehicle downtime and budget variance as first-order concerns. **Claim Frequency Rate** enables the benchmark comparison ("your rate vs portfolio average") that makes AFC's active management value immediately tangible.

## 5. Forecasting

### Approach

With ~30 customers averaging 2–4 claims per month each, per-customer time series models (ARIMA, Prophet) would be severely underfit — most customer-months contain 0 or 1 claims, and Prophet needs at least two full seasonal cycles to produce reliable intervals. Instead, all customers are pooled into a single **Poisson GLM**, which:

- handles count data natively (non-negative integer response)
- shares seasonal and trend signal across the full customer base
- gives interpretable coefficients and calibrated uncertainty

Claim counts are forecast first; costs are estimated in a second stage as predicted count × customer average cost per resolved claim, with partial pooling toward the global mean for customers with thin histories.

**Train**: Jan 2023 – Sep 2025 &nbsp;|&nbsp; **Test**: Oct – Dec 2025 &nbsp;|&nbsp; **Forecast**: Apr – Dec 2026  
Temporal split only — random row splits leak future information into the training set.

In [ ]:
TODAY      = pd.Timestamp('2026-03-09')
TRAIN_END  = pd.Timestamp('2025-09-30')
TEST_START = pd.Timestamp('2025-10-01')
TEST_END   = pd.Timestamp('2025-12-31')

hist = df[df['claim_date'] <= TODAY].copy()

# Complete customer x month panel so zero-claim months are explicit
all_months    = pd.date_range(
    hist['claim_date'].min().to_period('M').to_timestamp(),
    hist['claim_date'].max().to_period('M').to_timestamp(),
    freq='MS'
)
all_customers = hist['customer_id'].unique()

grid = (
    pd.MultiIndex.from_product([all_customers, all_months], names=['customer_id', 'month'])
    .to_frame(index=False)
)
grid['month'] = pd.to_datetime(grid['month'])

monthly_counts = (
    hist.assign(month=hist['claim_date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['customer_id', 'month'])['claim_id']
    .count()
    .reset_index(name='claim_count')
)

panel = (
    grid.merge(monthly_counts, on=['customer_id', 'month'], how='left')
    .fillna({'claim_count': 0})
)
panel['claim_count']   = panel['claim_count'].astype(int)
panel['month_of_year'] = panel['month'].dt.month
panel['year']          = panel['month'].dt.year
panel = panel.sort_values(['customer_id', 'month']).reset_index(drop=True)

# Rolling 12-month claim rate shifted by 1 to prevent leakage into the current month
customer_mean = panel.groupby('customer_id')['claim_count'].transform('mean')
panel['rolling_12m_rate'] = (
    panel.groupby('customer_id')['claim_count']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=3).mean())
)
panel['rolling_12m_rate'] = panel['rolling_12m_rate'].fillna(customer_mean)

le = LabelEncoder()
panel['customer_enc'] = le.fit_transform(panel['customer_id'])

panel.head()

In [ ]:
FEATURES = ['customer_enc', 'month_of_year', 'year', 'rolling_12m_rate']

train = panel[panel['month'] <= TRAIN_END].copy()
test  = panel[(panel['month'] >= TEST_START) & (panel['month'] <= TEST_END)].copy()

def make_X(df):
    X = df[FEATURES].copy().astype(float)
    X.insert(0, 'const', 1.0)
    return X

X_train, y_train = make_X(train), train['claim_count']
X_test,  y_test  = make_X(test),  test['claim_count']

glm    = sm.GLM(y_train, X_train, family=sm.families.Poisson())
result = glm.fit()

# Overdispersion check: residual deviance / df should be ~1 for well-specified Poisson
dispersion = result.deviance / result.df_resid
print(f'Dispersion statistic: {dispersion:.2f}')
if dispersion > 1.5:
    print('Overdispersion detected — refitting with Negative Binomial')
    result = sm.GLM(y_train, X_train, family=sm.families.NegativeBinomial()).fit()

print(result.summary())

In [ ]:
# Naive baseline: each customer's mean monthly claim count over the trailing 12 months of train
train_last12 = train[train['month'] > pd.Timestamp('2024-09-30')]
naive_rate   = train_last12.groupby('customer_id')['claim_count'].mean()
global_rate  = train['claim_count'].mean()

test = test.copy()
test['naive_pred'] = test['customer_id'].map(naive_rate).fillna(global_rate)
test['glm_pred']   = result.predict(X_test)

mae_naive = (test['claim_count'] - test['naive_pred']).abs().mean()
mae_glm   = (test['claim_count'] - test['glm_pred']).abs().mean()

pd.DataFrame({
    'Model'                          : ['Naive (12m avg)', 'Poisson GLM'],
    'MAE (claims / customer-month)'  : [round(mae_naive, 3), round(mae_glm, 3)],
})

In [ ]:
forecast_months = pd.date_range('2026-04-01', '2026-12-01', freq='MS')

forecast_grid = pd.DataFrame(
    [{'customer_id': c, 'month': m} for c in all_customers for m in forecast_months]
)
forecast_grid['month_of_year'] = forecast_grid['month'].dt.month
forecast_grid['year']          = forecast_grid['month'].dt.year
forecast_grid['customer_enc']  = le.transform(forecast_grid['customer_id'])

# Use each customer's last observed rolling rate as the forward-looking feature
last_rate = panel.groupby('customer_id')['rolling_12m_rate'].last()
forecast_grid['rolling_12m_rate'] = forecast_grid['customer_id'].map(last_rate)

X_fc = make_X(forecast_grid)
mu   = result.predict(X_fc).values

forecast_grid['predicted_claims'] = mu
# 80% prediction interval using exact Poisson quantiles
forecast_grid['ci_low']  = stats.poisson.ppf(0.10, np.maximum(mu, 0.01)).clip(min=0)
forecast_grid['ci_high'] = stats.poisson.ppf(0.90, np.maximum(mu, 0.01))

In [ ]:
# Stage 2: cost = predicted count x customer average cost per resolved claim
# Customers with thin history are shrunk toward the global mean (empirical Bayes, k=5)
resolved_cost   = resolved.groupby('customer_id')['actual_cost_eur'].agg(['mean', 'count'])
global_avg_cost = resolved['actual_cost_eur'].mean()
k = 5
resolved_cost['blended_cost'] = (
    (resolved_cost['mean'] * resolved_cost['count'] + global_avg_cost * k)
    / (resolved_cost['count'] + k)
)

forecast_grid['avg_cost'] = (
    forecast_grid['customer_id'].map(resolved_cost['blended_cost']).fillna(global_avg_cost)
)
forecast_grid['predicted_cost_eur'] = forecast_grid['predicted_claims'] * forecast_grid['avg_cost']

In [ ]:
annual = forecast_grid.groupby('customer_id').agg(
    predicted_claims    =('predicted_claims',    'sum'),
    predicted_cost_eur  =('predicted_cost_eur',  'sum'),
    ci_low              =('ci_low',              'sum'),
    ci_high             =('ci_high',             'sum'),
).round(1)

med_claims = annual['predicted_claims'].median()
med_cost   = annual['predicted_cost_eur'].median()

annual['risk_segment'] = annual.apply(
    lambda r: '{}/{}'.format(
        'high-freq' if r['predicted_claims']   >= med_claims else 'low-freq',
        'high-cost' if r['predicted_cost_eur'] >= med_cost   else 'low-cost'
    ),
    axis=1
)

annual.sort_values('predicted_cost_eur', ascending=False).style.format({
    'predicted_claims'   : '{:.0f}',
    'predicted_cost_eur' : '€{:,.0f}',
    'ci_low'             : '{:.0f}',
    'ci_high'            : '{:.0f}',
})

### Production

A monthly batch job retrains the GLM as new claims are added. The feature pipeline is a single SQL aggregation over the claims table — no streaming infrastructure is required at current scale. Forecast outputs are written to a table (`customer_id`, `month`, `predicted_claims`, `predicted_cost_eur`, `ci_low`, `ci_high`, `model_version`) consumed directly by the dashboard.

Monitoring: track actual vs predicted MAE on a rolling 3-month basis and alert if it exceeds 2× the test-set baseline. The most likely trigger is a customer significantly changing their fleet size, which the current model cannot observe directly.

New customers with fewer than 6 months of history fall back to the global mean rate by vehicle type until sufficient data accumulates.

### What would improve the model

- **Fleet size and composition per customer** — the most critical missing normaliser; claim counts scale with fleet size, which we currently cannot control for
- **Vehicle age and mileage** — strongest structural predictors of Motor and Fahrwerk claims
- **Weather data by region and date** — would explain Naturereignis spikes that currently appear as noise

## 6. Summary

Claim volume has been broadly stable across the observation period, with Motor and Karosserie damage dominating both frequency and cost. A small number of customers account for a disproportionate share of total cost exposure — these are the highest-leverage accounts for active fleet advisory and the clearest use case for AFC's managed services pitch.

The Poisson GLM provides customer-level claim and cost projections for April–December 2026 with 80% confidence intervals. Customers in the high-frequency / high-cost segment warrant priority attention for both loss prevention and commercial outreach. The main limitation is the absence of fleet size data: without normalising by vehicles managed, frequency comparisons across customers of different sizes remain approximate.